# Impedance ML Engineering — Session 14
## Refresh classification with an explicit teaching endpoint
**Core: 60 minutes · Classification · Optional extension: 20–40 minutes**

## Why this matters
Can current matrices forecast whether the next synthetic diagonal mean exceeds a fixed teaching cutoff?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Define a binary target without inventing clinical labels.
- Fit logistic and neural classifiers with matching patient partitions.
- Use logits, cross-entropy, and probability metrics correctly.
- Relate GLM coefficients to odds while separating inference from prediction.

## R-to-Python / prerequisite bridge
`glm(...,family=binomial)` → statsmodels GLM for inference; sklearn LogisticRegression for prediction; neural binary output is one logit.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
Logistic regression models log-odds as an affine function of inputs. Its inverse-link sigmoid yields a probability; exponentiating a coefficient gives a conditional odds ratio, not a risk ratio. A neural classifier learns a nonlinear logit mapping with a binary cross-entropy objective.

`BCEWithLogitsLoss` combines a numerically stable sigmoid/log-loss calculation. Pass raw logits and floating 0/1 targets of the same shape. ROC-AUC measures ranking; average precision summarizes precision/recall behavior; Brier score averages squared probability errors. A probability cutoff creates decisions and should reflect a declared objective.

Imports: `LogisticRegression` from `sklearn.linear_model`; `roc_auc_score`, `average_precision_score`, `brier_score_loss`, `confusion_matrix` from `sklearn.metrics`; `statsmodels.api as sm`, `statsmodels.formula.api as smf`; `torch`, `nn`.

## Task 1 — Specify the teaching target (5 min)
Set binary y to `(target_diag_mean_kohm > 7.0)`. The 7.0 kΩ cutoff is fixed for coding practice, with no clinical meaning. Predict class counts using train/validation only; report prevalence and unique patient counts. Explain why data-dependent thresholds must be fitted within development and frozen before evaluation.

## Task 2 — Establish classification baselines (6 min)
Use training prevalence as a constant probability baseline. Also classify persistence by current diagonal mean>7. Report validation confusion matrix at 0.5 and balanced accuracy; for constant probability report Brier score and average precision. Explain why ordinary accuracy alone can obscure class imbalance.

## Task 3 — Fit a logistic pipeline (8 min)
Use training-only numeric/categorical preprocessing and `LogisticRegression(C=1.0,max_iter=1000,random_state=42)`. Call `.predict_proba` and check class order via `.classes_`. Report validation ROC-AUC, average precision, Brier score, and confusion matrix. Both classes must exist for some metrics.

## Task 4 — Translate GLM interpretation (7 min)
On training pairs fit a binomial statsmodels GLM with current diagonal mean, planned horizon, and current protocol. Request patient-clustered covariance via `.fit(cov_type="cluster",cov_kwds={"groups":training_patient_ids})`. Report one coefficient and exponentiated odds ratio with interval. Interpret per-unit change without treating repeated pairs as independent or assuming causal meaning.

## Task 5 — Adapt the MLP to binary loss (8 min)
Use a small width 16 MLP with one output logit and `BCEWithLogitsLoss`. Target float32 shape must match (N, 1). Do not apply sigmoid before this loss. Apply sigmoid to detached outputs for probability reporting. Predict output/loss shapes before running.

## Task 6 — Train and compare fairly (10 min)
Train max 60 epochs with patient-separated validation and early stopping on validation BCE (patience 8), seed 42, Adam(lr=0.01). Compare logistic/MLP probabilities and metrics on identical validation rows. Class weighting is an explicit optional experiment, not an automatic improvement.

## Task 7 — Inspect errors by horizon (6 min)
At prespecified probability threshold 0.5, summarize false positives/negatives and counts by horizon. Plot probability distributions by observed class. Explain that ranking quality, probability accuracy, and threshold decisions are different objectives.

## Task 8 — Write a target and metric decision (2 min)
Document the synthetic endpoint, cutoff origin, loss, metrics, and class balance. State whether a classifier is justified over predicting the continuous value first. Clinical utility cannot be established by this coding endpoint.

## Retrieval practice
Audit `BCEWithLogitsLoss(torch.sigmoid(logit),label)` and an integer label tensor; describe the correct inputs.

## Professional reasoning
1. What could go wrong if a cutoff is chosen after looking at test outcomes?
2. Why is an odds ratio not a risk ratio?
3. Why can high ROC-AUC coexist with poor probability calibration?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Compare class_weight="balanced" logistic regression and weighted BCE using training class counts only. Inspect calibration as well as recall.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)